In [ ]:
import numpy as np
import soundfile as sf
import requests
import io
from IPython.display import Audio, display

# 1. Configuration
# Paste your generated public ngrok URL here (Ensure NO trailing slash and NO path suffix)
PUBLIC_API_URL = "https://complete-formerly-lab.ngrok-free.app"
TEST_AUDIO_FILE = "test_input_signal.wav"

print("⚙️ Step 1: Generating a clean 16kHz mono audio test file...")
# Create a 1-second synthetic 440Hz sine wave tone matching VibeASR's requirements
sample_rate = 16000
duration = 1.0  # seconds
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
synthetic_signal = 0.5 * np.sin(2 * np.pi * 440 * t)
sf.write(TEST_AUDIO_FILE, synthetic_signal, sample_rate)

print(f"📦 Step 2: Transmission phase. Dispatching payload to {PUBLIC_API_URL}/forward-speech...")
try:
    with open(TEST_AUDIO_FILE, "rb") as audio_payload:
        files = {"file": (TEST_AUDIO_FILE, audio_payload, "audio/wav")}
        data = {"voice": "af_sarah", "speed": "1.0"}

        # Injected critical headers to skip the ngrok browser warning walls
        headers = {
            "ngrok-skip-browser-warning": "true"
        }

        response = requests.post(
            f"{PUBLIC_API_URL}/forward-speech",
            files=files,
            data=data,
            headers=headers, # Passes intercept bypass matrix
            timeout=30       # Provides a safe processing window for CPU transcription & synthesis
        )

    # 3. Analyze Response Metadata
    if response.status_code == 200:
        print("\n✅ SUCCESS! Speech Forwarding Loop Complete.")

        # Extract the transcription text embedded in custom headers
        asr_transcription = response.headers.get("X-Transcribed-Text", "No header metadata found")
        print(f"📝 VibeASR.cpp Transcription Result: '{asr_transcription}'")

        print("🔊 Loading returned Kokoro TTS audio stream directly into your browser context:")
        # Read the raw audio bytes response payload directly into memory
        audio_bytes = io.BytesIO(response.content)
        display(Audio(audio_bytes.read(), rate=24000, autoplay=True))

    else:
        print(f"\n❌ Server rejected processing. Status Code: {response.status_code}")
        print(f"Error Detail: {response.text}")

except requests.exceptions.ConnectionError:
    print("\n❌ Connectivity Failure: Ensure your FastAPI backend server thread is active and the Ngrok URL matches exactly.")
except Exception as e:
    print(f"\n❌ Execution Exception: {str(e)}")
